# The Transformer Architecture & Attention
**Level:** Professional | **Interactive Machine Learning Curriculum**

> Scaled Dot-Product Attention, Multi-Head Attention, RoPE, ALiBi, FlashAttention, and Encoder/Decoder architectures.

---

### 📖 Conceptual Foundations

#### The Attention Breakthrough (Vaswani et al., 2017)

Introduced in the landmark paper *Attention Is All You Need*, the **Transformer** abandoned recurrent and convolutional connections entirely, relying solely on **Self-Attention** mechanisms. In self-attention, each token in a sequence directly computes pairwise interactions with every other token, capturing long-range contextual relationships with an $O(1)$ sequential path length and enabling massive parallel training on modern GPU clusters.


#### Scaled Dot-Product Attention Formula

Given packed matrices for Queries ($Q$), Keys ($K$), and Values ($V$), where $Q, K \in \mathbb{R}^{T \times d_k}$ and $V \in \mathbb{R}^{T \times d_v}$:

$$\text{Attention}(Q, K, V) = \text{softmax}\left( \frac{Q K^T}{\sqrt{d_k}} + M \right) V$$

- **Scaling Factor $\sqrt{d_k}$:** Prevents dot products from growing excessively large in high dimensions, which would push softmax into flat regions with vanishing gradients.
- **Mask Matrix $M$:** In autoregressive decoder models (e.g. GPT), an upper-triangular causal mask of $-\infty$ prevents positions from attending to future tokens.


#### Transformer Architecture Paradigms

| Paradigm | Attention Mechanism | Pioneering Models | Primary Strengths | Typical Applications |
| --- | --- | --- | --- | --- |
| Encoder-Only | Bidirectional Attention (all tokens attend to all tokens). | BERT, RoBERTa, DeBERTa | Deep bidirectional contextual representations. | Classification, Named Entity Recognition, Extractive QA, Embeddings. |
| Decoder-Only | Causal / Autoregressive Attention (tokens only attend to past tokens). | GPT-4, Llama 3, Mistral, Claude | Unbounded generative continuation, seamless zero-shot prompting. | Conversational agents, code generation, creative writing, reasoning. |
| Encoder-Decoder | Bidirectional Encoder + Causal Cross-Attention Decoder. | T5, BART, Original 2017 Transformer | Maps source sequence to arbitrary target sequence. | Machine translation, document summarization, question answering. |


#### Modern Positional Embeddings: RoPE & ALiBi

1. **Rotary Position Embedding (RoPE, Su et al., 2021):** Encodes absolute positions with a rotation matrix that naturally preserves relative positional distances in inner products: $q_m^T k_n = g(x_m, x_n, m - n)$. Standard in Llama, Mistral, and Gemma.
2. **ALiBi (Attention with Linear Biases, Press et al., 2022):** Injects a static linear bias penalty proportional to token distance $|i - j|$ directly into attention logits, enabling zero-shot context length extrapolation.


#### FlashAttention (Dao et al., 2022-2024)

Standard attention materializes an $O(N^2)$ attention matrix in high-bandwidth memory (HBM), creating a major memory bandwidth bottleneck. **FlashAttention** reorganizes the computation using tiling to compute softmax incrementally (online softmax) entirely within fast GPU SRAM without ever writing the full $N \times N$ intermediate attention matrix to HBM, achieving a **2x-4x speedup** and reducing memory from $O(N^2)$ to $O(N)$.



In [ ]:
# %matplotlib inline
import numpy as np

def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

def multi_head_attention(X, W_q, W_k, W_v, W_o, num_heads=2, causal_mask=True):
    # X: (T, d_model)
    T, d_model = X.shape
    d_k = d_model // num_heads
    
    # 1. Project into Q, K, V
    Q = np.dot(X, W_q).reshape(T, num_heads, d_k).transpose(1, 0, 2) # (H, T, d_k)
    K = np.dot(X, W_k).reshape(T, num_heads, d_k).transpose(1, 0, 2)
    V = np.dot(X, W_v).reshape(T, num_heads, d_k).transpose(1, 0, 2)
    
    # 2. Scaled Dot-Product Attention: (H, T, T)
    scores = np.matmul(Q, K.transpose(0, 2, 1)) / np.sqrt(d_k)
    
    if causal_mask:
        mask = np.triu(np.ones((T, T)), k=1)
        scores = np.where(mask == 1, -1e9, scores)
        
    attn_weights = softmax(scores, axis=-1)
    context = np.matmul(attn_weights, V) # (H, T, d_k)
    
    # 3. Concatenate heads and project
    context = context.transpose(1, 0, 2).reshape(T, d_model)
    output = np.dot(context, W_o)
    return output, attn_weights

# Test forward pass with T=4 tokens, d_model=8, 2 heads
np.random.seed(42)
T, d_model = 4, 8
X = np.random.randn(T, d_model)
W_q = np.random.randn(d_model, d_model) * 0.2
W_k = np.random.randn(d_model, d_model) * 0.2
W_v = np.random.randn(d_model, d_model) * 0.2
W_o = np.random.randn(d_model, d_model) * 0.2

out, weights = multi_head_attention(X, W_q, W_k, W_v, W_o, num_heads=2, causal_mask=True)
print(f'Multi-Head Output Shape: {out.shape}')
print('Head 0 Attention Weights (Causal Lower Triangular):')
print(np.round(weights[0], 3))


---

### 🎯 Hands-On Experiments & Challenges
1. **Experiment with parameters:** Modify the hyperparameters or dataset sizes in the cell above.
2. **Add visual diagnostics:** Plot loss curves or feature distributions using Matplotlib/Seaborn.
3. **Test with new data:** Try running the model on synthetic noisy data or custom inputs.


In [ ]:
# Your custom experiment code here:

